# Feedforward baselines from scratch

This notebook documents **binary classification** of campaign outcomes (successful vs failed) using the prepared **training** and **validation** tables under `data/features/`. The split is fixed; no additional holdout is created here.

Three small PyTorch models are trained and compared on the validation split at the end of the notebook.


In [10]:
import random
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from torch.utils.data import DataLoader, TensorDataset


## Data and labels

Predictor column names are read from `data/features/features_no_scale.txt` and `data/features/features_scale.txt`.

Observations are restricted to `successful` or `failed`. The text field concatenates `name` and `blurb`. The target is coded **1** for successful campaigns and **0** otherwise.


In [11]:
def load_feature_names(path):
    with open(path) as f:
        return [line.strip() for line in f if line.strip()]


FEATURES_NO_SCALE = load_feature_names("data/features/features_no_scale.txt")
FEATURES_SCALE = load_feature_names("data/features/features_scale.txt")


def prepare_aligned(df):
    df = df[df["state"].isin(["successful", "failed"])].copy()
    df["text"] = df["name"].fillna("") + " " + df["blurb"].fillna("")
    df["label"] = (df["state"] == "successful").astype(int)
    df = df.dropna(subset=["text", "label"]).reset_index(drop=True)
    return df


train_aligned = prepare_aligned(pd.read_csv("data/features/train.csv"))
val_aligned = prepare_aligned(pd.read_csv("data/features/val.csv"))

print("Train rows:", len(train_aligned), "Val rows:", len(val_aligned))


Train rows: 66096 Val rows: 22032


## Tabular input matrix

The no-scale and scale blocks are concatenated column-wise. A `StandardScaler` is **fit on the training partition only** and applied to the validation partition. `nan_to_num` replaces non-finite values before modeling.


In [12]:
def build_feature_matrix(aligned_df, scaler=None, *, fit_scaler):
    X_no = aligned_df[FEATURES_NO_SCALE].astype(np.float64).to_numpy()
    X_scale_raw = aligned_df[FEATURES_SCALE].astype(np.float64).to_numpy()
    if fit_scaler:
        scaler = StandardScaler()
        X_scale = scaler.fit_transform(X_scale_raw)
    else:
        assert scaler is not None
        X_scale = scaler.transform(X_scale_raw)
    X = np.hstack([X_no, X_scale])
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    return X.astype(np.float32), scaler


X_train, scaler = build_feature_matrix(train_aligned, fit_scaler=True)
X_val, _ = build_feature_matrix(val_aligned, scaler=scaler, fit_scaler=False)

y_train = train_aligned["label"].to_numpy(dtype=np.int64)
y_val = val_aligned["label"].to_numpy(dtype=np.int64)

input_dim = X_train.shape[1]
print("Input dim:", input_dim)


Input dim: 29


## Model 1: structured features only

A multilayer perceptron (`Linear` → `ReLU` → `Dropout`) maps the 29-dimensional tabular vector to two logits. Training uses AdamW and cross-entropy loss; no pretrained components are used.


In [13]:
class TabularMLP(nn.Module):
    def __init__(self, in_dim, hidden_dims=(128, 64), num_classes=2, dropout=0.2):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


## Training (structured only)

Optimization: AdamW with cross-entropy loss for **30** epochs. Training and validation loss are printed periodically (first epoch, every fifth epoch, and the final epoch).


In [14]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def train_mlp(
    X_train,
    y_train,
    X_val,
    y_val,
    *,
    epochs=30,
    batch_size=256,
    lr=1e-3,
    weight_decay=1e-4,
    hidden_dims=(128, 64),
    dropout=0.2,
    seed=42,
):
    set_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_ds = TensorDataset(
        torch.from_numpy(X_train),
        torch.from_numpy(y_train),
    )
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=False)

    model = TabularMLP(X_train.shape[1], hidden_dims=hidden_dims, dropout=dropout).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.CrossEntropyLoss()

    X_val_t = torch.from_numpy(X_val).to(device)
    y_val_t = torch.from_numpy(y_val).to(device)

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        n_batches = 0
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            opt.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            opt.step()
            total_loss += float(loss.detach().cpu())
            n_batches += 1

        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_t)
            val_loss = float(loss_fn(val_logits, y_val_t).cpu())
        if epoch == 1 or epoch % 5 == 0 or epoch == epochs:
            print(
                f"epoch {epoch:03d}  train_loss={total_loss / max(n_batches, 1):.4f}  val_loss={val_loss:.4f}"
            )

    model.eval()
    with torch.no_grad():
        val_logits = model(X_val_t)
        val_probs = torch.softmax(val_logits, dim=-1)[:, 1].cpu().numpy()
        val_pred = torch.argmax(val_logits, dim=-1).cpu().numpy()

    metrics = {
        "accuracy": float(accuracy_score(y_val, val_pred)),
        "f1": float(f1_score(y_val, val_pred)),
        "precision": float(precision_score(y_val, val_pred)),
        "recall": float(recall_score(y_val, val_pred)),
        "roc_auc": float(roc_auc_score(y_val, val_probs)),
    }
    print("Validation:", metrics)
    return model, metrics, device


print("structured features only")
tabular_model, tabular_metrics, _ = train_mlp(X_train, y_train, X_val, y_val)


structured features only
epoch 001  train_loss=0.5785  val_loss=0.5546
epoch 005  train_loss=0.5360  val_loss=0.5346
epoch 010  train_loss=0.5211  val_loss=0.5248
epoch 015  train_loss=0.5125  val_loss=0.5170
epoch 020  train_loss=0.5037  val_loss=0.5118
epoch 025  train_loss=0.4947  val_loss=0.5030
epoch 030  train_loss=0.4886  val_loss=0.4979
Validation: {'accuracy': 0.7485021786492375, 'f1': 0.7960242959690779, 'precision': 0.7669716961055544, 'recall': 0.8273645546372819, 'roc_auc': 0.8272775615948814}


## Text as integer sequences

Text is lowercased and split on whitespace into tokens. A vocabulary of the **12 000** most frequent training tokens is built (plus `<unk>` and `<pad>`). Each sequence is padded or truncated to **96** positions. The validation set uses the **training** vocabulary; tokens absent from that vocabulary map to `<unk>`.


In [15]:
MAX_LEN = 96
MAX_VOCAB_WORDS = 12000


def build_word_vocab(texts, max_words=MAX_VOCAB_WORDS):
    counts = Counter()
    for t in texts:
        counts.update(str(t).lower().split())
    top = [w for w, _ in counts.most_common(max_words)]
    word2idx = {"<pad>": 0, "<unk>": 1}
    for w in top:
        word2idx[w] = len(word2idx)
    return word2idx, len(word2idx)


def texts_to_ids(texts, word2idx, max_len=MAX_LEN):
    pad = word2idx["<pad>"]
    unk = word2idx["<unk>"]
    out = np.full((len(texts), max_len), pad, dtype=np.int64)
    for i, t in enumerate(texts):
        ids = [word2idx.get(w, unk) for w in str(t).lower().split()][:max_len]
        if ids:
            out[i, : len(ids)] = ids
    return out


word2idx, vocab_size = build_word_vocab(train_aligned["text"].tolist())
train_ids = texts_to_ids(train_aligned["text"].tolist(), word2idx)
val_ids = texts_to_ids(val_aligned["text"].tolist(), word2idx)
print("Vocab size:", vocab_size, "| train_ids:", train_ids.shape)


Vocab size: 12002 | train_ids: (66096, 96)


## Models 2 and 3: text only, and text with tabular inputs

Each architecture applies a trainable **embedding** to token indices, **mean-pools** over non-padding tokens, and passes the result through an MLP. The third model **concatenates** the pooled text vector with the **tabular feature vector** (same construction as Model 1) before the MLP head.


In [16]:
class TextEmbeddingMLP(nn.Module):
    def __init__(
        self,
        vocab_size,
        embed_dim=96,
        hidden_dims=(128, 64),
        num_classes=2,
        dropout=0.2,
        pad_idx=0,
    ):
        super().__init__()
        self.pad_idx = pad_idx
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        layers = []
        prev = embed_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, num_classes))
        self.head = nn.Sequential(*layers)

    def forward(self, token_ids):
        emb = self.embed(token_ids)
        mask = (token_ids != self.pad_idx).float().unsqueeze(-1)
        summed = (emb * mask).sum(dim=1)
        denom = mask.sum(dim=1).clamp(min=1.0)
        pooled = summed / denom
        return self.head(pooled)


class TextStructuredMLP(nn.Module):
    def __init__(
        self,
        vocab_size,
        tab_dim,
        embed_dim=96,
        hidden_dims=(128, 64),
        num_classes=2,
        dropout=0.2,
        pad_idx=0,
    ):
        super().__init__()
        self.pad_idx = pad_idx
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        layers = []
        prev = embed_dim + tab_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, num_classes))
        self.head = nn.Sequential(*layers)

    def forward(self, token_ids, x_tab):
        emb = self.embed(token_ids)
        mask = (token_ids != self.pad_idx).float().unsqueeze(-1)
        summed = (emb * mask).sum(dim=1)
        denom = mask.sum(dim=1).clamp(min=1.0)
        pooled = summed / denom
        h = torch.cat([pooled, x_tab], dim=1)
        return self.head(h)


## Training the text models

Optimization matches the structured-only setup (AdamW, cross-entropy). Validation loss and final predictions are computed in **mini-batches** over the validation set to reduce peak device memory.


In [17]:
def train_text_classifier(
    model,
    train_ids,
    y_train,
    val_ids,
    y_val,
    X_train_tab=None,
    X_val_tab=None,
    *,
    name="model",
    epochs=20,
    batch_size=256,
    lr=1e-3,
    weight_decay=1e-4,
    seed=42,
):
    set_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    has_tab = X_train_tab is not None

    if has_tab:
        train_ds = TensorDataset(
            torch.from_numpy(train_ids),
            torch.from_numpy(X_train_tab.astype(np.float32)),
            torch.from_numpy(y_train),
        )
    else:
        train_ds = TensorDataset(torch.from_numpy(train_ids), torch.from_numpy(y_train))

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=False)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.CrossEntropyLoss()

    def val_loss_batched():
        model.eval()
        total, n = 0.0, 0
        with torch.no_grad():
            for i in range(0, len(val_ids), 2048):
                sl = slice(i, i + 2048)
                vb = torch.from_numpy(val_ids[sl]).to(device)
                yb = torch.from_numpy(y_val[sl]).to(device)
                if has_tab:
                    tb = torch.from_numpy(X_val_tab[sl].astype(np.float32)).to(device)
                    logits = model(vb, tb)
                else:
                    logits = model(vb)
                bs = yb.shape[0]
                total += float(loss_fn(logits, yb).cpu()) * bs
                n += bs
        return total / max(n, 1)

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        n_batches = 0
        for batch in train_loader:
            if has_tab:
                ids, tab, yb = batch
                ids = ids.to(device)
                tab = tab.to(device)
                yb = yb.to(device)
                logits = model(ids, tab)
            else:
                ids, yb = batch
                ids = ids.to(device)
                yb = yb.to(device)
                logits = model(ids)
            opt.zero_grad(set_to_none=True)
            loss = loss_fn(logits, yb)
            loss.backward()
            opt.step()
            total_loss += float(loss.detach().cpu())
            n_batches += 1

        model.eval()
        val_loss = val_loss_batched()
        if epoch == 1 or epoch % 5 == 0 or epoch == epochs:
            print(
                f"[{name}] epoch {epoch:03d}  train_loss={total_loss / max(n_batches, 1):.4f}  val_loss={val_loss:.4f}"
            )

    model.eval()
    pred_chunks = []
    prob_chunks = []
    with torch.no_grad():
        for i in range(0, len(val_ids), 2048):
            sl = slice(i, i + 2048)
            vb = torch.from_numpy(val_ids[sl]).to(device)
            if has_tab:
                tb = torch.from_numpy(X_val_tab[sl].astype(np.float32)).to(device)
                logits = model(vb, tb)
            else:
                logits = model(vb)
            pred_chunks.append(torch.argmax(logits, dim=-1).cpu().numpy())
            prob_chunks.append(torch.softmax(logits, dim=-1)[:, 1].cpu().numpy())
    val_pred = np.concatenate(pred_chunks, axis=0)
    val_probs = np.concatenate(prob_chunks, axis=0)

    metrics = {
        "accuracy": float(accuracy_score(y_val, val_pred)),
        "f1": float(f1_score(y_val, val_pred)),
        "precision": float(precision_score(y_val, val_pred)),
        "recall": float(recall_score(y_val, val_pred)),
        "roc_auc": float(roc_auc_score(y_val, val_probs)),
    }
    print(f"[{name}] Validation:", metrics)
    return model, metrics, device


print("text only")
text_model = TextEmbeddingMLP(vocab_size=vocab_size, embed_dim=96, hidden_dims=(128, 64), dropout=0.2)
text_model, text_metrics, _ = train_text_classifier(
    text_model,
    train_ids,
    y_train,
    val_ids,
    y_val,
    name="text_only",
    epochs=20,
    batch_size=256,
)

print("text + structured")
hybrid_model = TextStructuredMLP(
    vocab_size=vocab_size,
    tab_dim=X_train.shape[1],
    embed_dim=96,
    hidden_dims=(128, 64),
    dropout=0.2,
)
hybrid_model, hybrid_metrics, _ = train_text_classifier(
    hybrid_model,
    train_ids,
    y_train,
    val_ids,
    y_val,
    X_train_tab=X_train,
    X_val_tab=X_val,
    name="text+structured",
    epochs=20,
    batch_size=256,
)


text only
[text_only] epoch 001  train_loss=0.6236  val_loss=0.5916
[text_only] epoch 005  train_loss=0.4829  val_loss=0.5571
[text_only] epoch 010  train_loss=0.4002  val_loss=0.6134
[text_only] epoch 015  train_loss=0.3147  val_loss=0.7603
[text_only] epoch 020  train_loss=0.2302  val_loss=1.0019
[text_only] Validation: {'accuracy': 0.6899055918663762, 'f1': 0.7440815103386275, 'precision': 0.7287936601115351, 'recall': 0.7600244872972146, 'roc_auc': 0.7320630769856968}
text + structured
[text+structured] epoch 001  train_loss=0.5548  val_loss=0.5190
[text+structured] epoch 005  train_loss=0.4165  val_loss=0.4867
[text+structured] epoch 010  train_loss=0.3283  val_loss=0.5459
[text+structured] epoch 015  train_loss=0.2516  val_loss=0.6795
[text+structured] epoch 020  train_loss=0.1806  val_loss=0.9174
[text+structured] Validation: {'accuracy': 0.7446441539578794, 'f1': 0.7914133175144594, 'precision': 0.7676208285385501, 'recall': 0.816727884909703, 'roc_auc': 0.803249874988851}


## Comparison

The table below reports **accuracy**, **F1**, **precision**, **recall**, and **ROC AUC** (probability of the positive class from softmax, where applicable) on the validation partition. Rows are ordered by **F1** (descending).


In [18]:
comparison_df = pd.DataFrame(
    [
        {"Model": "Structured features only (MLP)", **tabular_metrics},
        {"Model": "Text only (embedding + mean-pool + MLP)", **text_metrics},
        {"Model": "Text + structured (embedding + mean-pool + MLP)", **hybrid_metrics},
    ]
)
comparison_df = comparison_df[
    ["Model", "accuracy", "f1", "precision", "recall", "roc_auc"]
].sort_values("f1", ascending=False).reset_index(drop=True)
comparison_df


,Model,accuracy,f1,precision,recall,roc_auc
0,Structured features only (MLP),0.748502,0.796024,0.766972,0.827365,0.827278
1,Text + structured (embedding + mean-pool + MLP),0.744644,0.791413,0.767621,0.816728,0.803250
2,Text only (embedding + mean-pool + MLP),0.689906,0.744082,0.728794,0.760024,0.732063
